# Query the lakehouse with Trino

This notebook connects to **Trino** to query the same Iceberg tables that
Spark wrote in the medallion pipeline. The key teaching point: Spark (batch
ETL) and Trino (interactive SQL) share one catalog and one copy of data —
the decoupled-compute property that defines a lakehouse.

> **Prerequisites:** `make pipeline` and `make serving` must have run.
> The `05-query-with-trino.ipynb` notebook runs *inside* the spark-iceberg
> pod, so we connect to Trino over in-cluster DNS at `trino:8080`.

## 1. Install the Trino Python client

The Trino Python DB-API driver is not pre-installed in the spark-iceberg
image. We install it now (it's lightweight — no JVM involved).

In [ ]:
%pip install trino --quiet

## 2. Connect to Trino

Trino exposes the Iceberg REST catalog as the `iceberg` catalog. Namespaces
(`bronze`, `silver`, `gold`) appear as Trino schemas.

In [ ]:
from trino.dbapi import Connection

conn = Connection(
    host="trino",
    port=8080,
    catalog="iceberg",
)
cur = conn.cursor()
print("Connected to Trino")

# Verify the catalog is visible
cur.execute("SHOW CATALOGS")
catalogs = [row[0] for row in cur.fetchall()]
print(f"Catalogs: {catalogs}")

## 3. Explore the medallion namespaces

Bronze, silver, and gold — the same tables Spark created in the pipeline.

In [ ]:
cur.execute("SHOW SCHEMAS FROM iceberg")
schemas = [row[0] for row in cur.fetchall()]
print(f"Iceberg namespaces (Trino schemas): {schemas}")

In [ ]:
cur.execute("SHOW TABLES FROM iceberg.gold")
gold_tables = [row[0] for row in cur.fetchall()]
print(f"Gold-layer tables: {gold_tables}")

## 4. Query the gold table

The `item_performance` table was built by Spark's
`40_silver_to_gold.py` / notebook `04`. Trino reads the same Parquet files.

In [ ]:
cur.execute("SELECT * FROM iceberg.gold.item_performance LIMIT 10")
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(f"Columns: {columns}")
print()
for row in rows:
    print(row)

## 5. Aggregate queries

Trino's distributed SQL engine handles aggregations that would be wasteful
in Spark (no need to start a full SparkSession for a quick count).

In [ ]:
cur.execute("SELECT COUNT(*) AS row_count FROM iceberg.gold.item_performance")
print(f"Total rows in item_performance: {cur.fetchone()[0]}")

In [ ]:
cur.execute("""
    SELECT category, COUNT(*) AS cnt, ROUND(AVG(revenue), 2) AS avg_revenue
    FROM iceberg.gold.item_performance
    GROUP BY category
    ORDER BY avg_revenue DESC
""")
rows = cur.fetchall()
columns = [desc[0] for desc in cur.description]
print(" | ".join(columns))
print("-" * 50)
for row in rows:
    print(" | ".join(str(v) for v in row))

## 6. Verify it's the same data as Spark

Compare with what Spark sees. If the pipeline was run, the counts match.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
spark_count = spark.sql("SELECT COUNT(*) AS cnt FROM demo.gold.item_performance").collect()[0][0]
trino_count = conn.cursor().execute("SELECT COUNT(*) FROM iceberg.gold.item_performance").fetchone()[0]
print(f"Spark row count:  {spark_count}")
print(f"Trino row count:  {trino_count}")
print(f"Match: {spark_count == trino_count}")

## Key takeaway

Spark and Trino access the **same** Iceberg tables through the **same** REST
catalog. There is no data movement, no export step, no duplication. This is
the lakehouse value proposition:

- **Spark** for batch ETL (the medallion pipeline)
- **Trino** for interactive SQL, ad-hoc queries, and BI
- **Metabase** (or any BI tool) connects through Trino

One copy of data, two compute engines, one catalog.